# MVP – Machine Learning & Analytics
**Nome:** João Victor de Assis Natividade
**Matrícula:** 4052025002212
**Dataset:** [Open-Meteo Historical Weather API](https://open-meteo.com/) — dados de reanálise ERA5, públicos (CC BY 4.0), sem chave/login

# Descrição do Problema
O ajuste de prazos em contratos sensíveis ao clima depende de saber, com antecedência,
se vai chover. Este projeto prevê a **ocorrência de chuva no dia seguinte** (sim/não)
na cidade de **Lajeado/RS**, a partir de variáveis meteorológicas observadas no dia
corrente. É a mesma motivação do meu TCC (plataforma Ricarol), onde a previsão de chuva
personaliza automaticamente o prazo de contratos.

O foco do MVP **não é obter o melhor modelo possível**, e sim demonstrar um fluxo de ML
coerente, reprodutível, honesto e bem justificado, do problema à conclusão.

## Hipóteses do Problema
As hipóteses que tracei para guiar a análise exploratória são as seguintes:
1. **A umidade do dia é o preditor isolado mais forte para a chuva do dia seguinte?**
2. **Existe um padrão sazonal claro, com meses sistematicamente mais chuvosos que outros?**
3. **A persistência ("se choveu hoje, tende a chover amanhã") é um baseline difícil de superar?**

## Tipo de Problema
Este é um problema de **aprendizado supervisionado** (classificação binária), pois temos
uma variável alvo bem definida (`rain_tomorrow`), onde 1 representa chuva no dia seguinte
e 0 representa ausência de chuva. Como os dados formam uma **série temporal**, toda a
divisão treino/teste respeita a ordem cronológica (treina no passado, testa no futuro);
nunca embaralhamos as datas.

## Seleção de Dados
Os dados vêm da API pública de arquivo do Open-Meteo (reanálise ERA5), de domínio público,
carregados diretamente por URL — **sem upload, login, token ou chave**, de forma que o
notebook roda do início ao fim. A premissa é que as condições meteorológicas observadas
(precipitação, temperaturas, umidade, pressão e vento), somadas a sinais de sazonalidade,
são preditivas para o curto prazo. Uma restrição inerente é que a previsão de chuva a um dia,
usando apenas dados diários agregados de uma única localidade, é um problema **difícil** —
portanto métricas modestas são esperadas e fazem parte de uma avaliação honesta.

## Atributos do Dataset
As variáveis brutas obtidas da API são:
- `date`: data da observação diária
- `tavg`, `tmin`, `tmax`: temperaturas média, mínima e máxima (°C)
- `precip`: precipitação acumulada no dia (mm)
- `humidity`: umidade relativa média (%)
- `pressure`: pressão média à superfície (hPa)
- `wind_speed`: velocidade máxima do vento a 10 m (km/h)

A partir delas derivamos o alvo `rain_tomorrow` e features de sazonalidade e memória
(descritas na seção de preparação).


## 2. Configuração do ambiente e reprodutibilidade

Fixamos as *seeds* para garantir que o notebook produza resultados reproduzíveis a cada execução.

In [ ]:
# Bibliotecas padrão do ecossistema científico Python
import os, random, time
_T0 = time.time()  # marca o início para medir o tempo total de execução
# Reprodutibilidade total: definir ANTES de importar o TensorFlow
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_auc_score, roc_curve)

import tensorflow as tf

# Fixa todas as fontes de aleatoriedade (Python, NumPy e TensorFlow)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)      # garante reprodutibilidade do Keras
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
print("Ambiente pronto. TensorFlow:", tf.__version__)

## 3. Apresentação e carga dos dados

**Fonte.** [Open-Meteo Historical Weather API](https://open-meteo.com/) — dados de
reanálise ERA5, públicos, licença CC BY 4.0, **sem necessidade de chave de API,
login ou upload**. O notebook pode ser executado do início ao fim pelo professor.

**Local.** Lajeado/RS (lat −29.4669, lon −51.9644).

**Período.** 01/01/2019 a 31/05/2026 (~2.708 observações diárias, ~7,4 anos).

**Variáveis solicitadas (todas reais, nada sintético):** temperatura média/mín/máx,
precipitação acumulada, umidade relativa média, pressão à superfície média e
velocidade máxima do vento.


In [ ]:
# URL pública da API de arquivo do Open-Meteo.
# Todas as variáveis abaixo são REAIS e retornadas pelo endpoint diário —
# não há geração de dados sintéticos/mock.
URL = (
    "https://archive-api.open-meteo.com/v1/archive?"
    "latitude=-29.4669&longitude=-51.9644"
    "&start_date=2019-01-01&end_date=2026-05-31"
    "&daily=temperature_2m_mean,temperature_2m_min,temperature_2m_max,"
    "precipitation_sum,relative_humidity_2m_mean,"
    "surface_pressure_mean,wind_speed_10m_max"
    "&timezone=America%2FSao_Paulo"
)

resp = requests.get(URL, timeout=60)
resp.raise_for_status()
daily = resp.json()["daily"]

df = pd.DataFrame({
    "date":       pd.to_datetime(daily["time"]),
    "tavg":       daily["temperature_2m_mean"],
    "tmin":       daily["temperature_2m_min"],
    "tmax":       daily["temperature_2m_max"],
    "precip":     daily["precipitation_sum"],
    "humidity":   daily["relative_humidity_2m_mean"],
    "pressure":   daily["surface_pressure_mean"],
    "wind_speed": daily["wind_speed_10m_max"],
}).sort_values("date").reset_index(drop=True)

print(f"Registros: {len(df)}  |  Período: {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

## 4. Análise exploratória dos dados (EDA)

Antes de modelar, precisamos entender a estrutura, a qualidade e a distribuição dos dados.

In [ ]:
# Tipos e valores ausentes
print("Tipos das variáveis:")
print(df.dtypes, "\n")
print("Valores ausentes por coluna:")
print(df.isna().sum())

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe().T

### 4.1. Definição da variável-alvo e verificação de desbalanceamento

Criamos o alvo `rain_tomorrow`: **1 se a precipitação do dia seguinte for > 1 mm**.
O limiar de 1 mm descarta "traços" de chuva (orvalho, garoa irrelevante) que poluem
o sinal. Em seguida verificamos o balanceamento entre as classes — fundamental para
escolher métricas adequadas.

In [ ]:
LIMIAR_CHUVA = 1.0  # mm

df["rain_today"]    = (df["precip"] > LIMIAR_CHUVA).astype(int)
df["rain_tomorrow"] = df["rain_today"].shift(-1)   # alvo = chuva do DIA SEGUINTE

# A última linha não tem "amanhã" observado -> removida
df = df.dropna(subset=["rain_tomorrow"]).copy()
df["rain_tomorrow"] = df["rain_tomorrow"].astype(int)

dist = df["rain_tomorrow"].value_counts(normalize=True).rename({0: "Não chove", 1: "Chove"})
print("Distribuição da variável-alvo (amanhã):")
print((dist * 100).round(1).astype(str) + " %")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["rain_tomorrow"].value_counts().sort_index().plot(
    kind="bar", ax=ax[0], color=["#4C72B0", "#55A868"])
ax[0].set_xticklabels(["Não chove (0)", "Chove (1)"], rotation=0)
ax[0].set_title("Contagem por classe (alvo = amanhã)")
ax[0].set_ylabel("nº de dias")

# Sazonalidade: proporção de dias de chuva por mês
df["month"] = df["date"].dt.month
(df.groupby("month")["rain_today"].mean() * 100).plot(
    kind="bar", ax=ax[1], color="#4C72B0")
ax[1].set_title("Frequência de chuva por mês (%)")
ax[1].set_xlabel("mês"); ax[1].set_ylabel("% de dias com chuva")
plt.tight_layout(); plt.show()

**Análise:** As classes ficam moderadamente desbalanceadas: **61,8% dos dias não
têm chuva no dia seguinte e 38,2% têm**. Há uma classe majoritária ("não chove"), o que
é um ponto de atenção crítico: **acurácia sozinha engana** — um modelo que sempre prevê
"não chove" acertaria ~62% dos casos sem prever uma única chuva (confirmaremos isso no
baseline de classe majoritária). Por isso, avaliaremos também precisão, recall, F1 e
matriz de confusão. O gráfico de sazonalidade já dá um indício a favor da **Hipótese 2**:
a frequência de chuva varia claramente entre os meses (de ~26% em julho a ~48% em janeiro),
justificando o uso de features sazonais (seno/cosseno).

In [ ]:
# Matriz de correlação entre variáveis contínuas e a chuva de amanhã
import numpy as np
cols_corr = ["precip","tavg","tmin","tmax","humidity","pressure","wind_speed","rain_tomorrow"]
corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_corr))); ax.set_xticklabels(cols_corr, rotation=45, ha="right")
ax.set_yticks(range(len(cols_corr))); ax.set_yticklabels(cols_corr)
for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Correlação de Pearson"); plt.tight_layout(); plt.show()

**Análise:** Nenhuma variável isolada tem correlação forte com a chuva de amanhã —
o que é esperado e reforça que o problema é difícil, exigindo um modelo que combine vários
sinais. Olhando a correlação de cada variável com `rain_tomorrow`, as mais relevantes (em
valor absoluto) são a **pressão (−0,31)** e a **temperatura mínima (+0,27)**, seguidas da
precipitação (+0,24) e da temperatura média (+0,23); a **umidade fica em +0,15**. Esse
resultado oferece um teste direto à **Hipótese 1**: ao contrário do que eu supunha, a
umidade **não** é o preditor isolado mais forte — a pressão à superfície e a temperatura
mínima a superam. A baixa correlação geral também explica por que, mais à frente, modelos
mais flexíveis (rede neural) levam vantagem ao capturar relações não lineares.

## 5. Preparação dos dados e engenharia de atributos

**Decisões e justificativas:**

1. **Features de sazonalidade (`sin`, `cos`).** O dia do ano é cíclico: 31/dez e
   1/jan são vizinhos. Codificar o dia com seno e cosseno preserva essa continuidade,
   permitindo à rede aprender padrões anuais sem uma "quebra" artificial entre dezembro
   e janeiro.

2. **Feature de memória (`lag_precip`, `lag_humidity`).** O clima tem inércia: o que
   ocorreu **hoje** ajuda a prever **amanhã**. Incluímos a precipitação e a umidade do
   próprio dia como preditores legítimos (são observados *antes* do alvo, então não há
   vazamento).

3. **Sem features sintéticas.** Toda feature ou é uma medição real, ou é uma
   transformação determinística de uma medição (sazonalidade, defasagem).

4. **Normalização via `StandardScaler`**, mas ajustada **somente no treino** (ver
   seção 6) para evitar vazamento de dados.

> **Nota.** Todas as features ou são medições reais da API, ou transformações
> determinísticas dessas medições (sazonalidade e defasagens). Não há features
> sintéticas/aleatórias, garantindo que cada variável tenha justificativa física.


In [ ]:
# --- Sazonalidade contínua ---
doy = df["date"].dt.dayofyear
df["sin"] = np.sin(2 * np.pi * doy / 365.25)
df["cos"] = np.cos(2 * np.pi * doy / 365.25)

# --- Memória de curto prazo (valores do PRÓPRIO dia, observados antes do alvo) ---
df["lag_precip"]   = df["precip"]      # chuva de hoje ajuda a prever amanhã
df["lag_humidity"] = df["humidity"]

FEATURES = ["tavg","tmin","tmax","precip","humidity","pressure",
            "wind_speed","sin","cos","lag_precip","lag_humidity"]
TARGET = "rain_tomorrow"

print("Features usadas:", FEATURES)
df[FEATURES + [TARGET]].head()

## 6. Divisão dos dados (respeitando o tempo)

Por ser série temporal, **não** podemos embaralhar. Treinamos no passado e testamos
no futuro. Reservamos os **últimos ~12 meses** como conjunto de teste (dados nunca
vistos) e usamos o restante para treino/validação.

A normalização (`StandardScaler`) é **ajustada apenas no treino** e depois aplicada
ao teste — assim o conjunto de teste não "vaza" informação para o pré-processamento.

In [ ]:
# Corte temporal: teste = últimos 12 meses
data_corte = df["date"].max() - pd.DateOffset(months=12)

train = df[df["date"] <= data_corte].copy()
test  = df[df["date"] >  data_corte].copy()

X_train, y_train = train[FEATURES].values, train[TARGET].values
X_test,  y_test  = test[FEATURES].values,  test[TARGET].values

print(f"Treino: {len(train)} dias ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Teste : {len(test)} dias ({test['date'].min().date()} → {test['date'].max().date()})")
print(f"% chuva no treino: {y_train.mean():.1%}  |  % chuva no teste: {y_test.mean():.1%}")

# Normalização ajustada SÓ no treino
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

## 7. Baseline ingênuo

Antes de qualquer modelo elaborado, definimos uma referência. Dois baselines simples,
naturais para chuva:

- **Persistência:** "amanhã faz o mesmo que hoje" (`rain_tomorrow = rain_today`).
- **Classe majoritária:** prever sempre a classe mais comum.

Se os modelos de ML não superarem esses baselines, eles não estão agregando valor.

In [ ]:
def avaliar(nome, y_true, y_pred, y_prob=None):
    out = {
        "modelo": nome,
        "acuracia":  accuracy_score(y_true, y_pred),
        "precisao":  precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }
    out["auc"] = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    return out

resultados = []

# Baseline 1: persistência (amanhã = hoje)
pred_persist = test["rain_today"].values
resultados.append(avaliar("Baseline: persistência", y_test, pred_persist))

# Baseline 2: classe majoritária
classe_maj = int(round(y_train.mean()))
pred_maj = np.full_like(y_test, classe_maj)
resultados.append(avaliar("Baseline: classe majoritária", y_test, pred_maj))

pd.DataFrame(resultados).set_index("modelo")

## 8. Modelagem: modelos candidatos

Treinamos quatro abordagens, das mais simples às mais flexíveis:

1. **Regressão Logística** — modelo linear, interpretável, ótimo ponto de partida.
2. **Random Forest** — ensemble de árvores por *bagging*, captura interações não lineares.
3. **Gradient Boosting (HistGradientBoosting)** — ensemble de árvores por *boosting*,
   costuma ser forte em dados tabulares e serve de contraponto à Random Forest.
4. **Rede Neural Densa (MLP, TensorFlow)** — a mesma família usada no meu TCC.

Todos usam `class_weight`/balanceamento para lidar com a classe majoritária, e a
mesma normalização. A comparação é justa: mesmos dados de treino e teste, mesmas
features.

In [ ]:
# --- Modelo 1: Regressão Logística ---
logit = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
logit.fit(X_train_s, y_train)
p_logit = logit.predict(X_test_s)
prob_logit = logit.predict_proba(X_test_s)[:, 1]
resultados.append(avaliar("Regressão Logística", y_test, p_logit, prob_logit))

# --- Modelo 2: Random Forest ---
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train_s, y_train)
p_rf = rf.predict(X_test_s)
prob_rf = rf.predict_proba(X_test_s)[:, 1]
resultados.append(avaliar("Random Forest", y_test, p_rf, prob_rf))

# --- Modelo 3: Gradient Boosting ---
gb = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)
gb.fit(X_train_s, y_train)
p_gb = gb.predict(X_test_s)
prob_gb = gb.predict_proba(X_test_s)[:, 1]
resultados.append(avaliar("Gradient Boosting", y_test, p_gb, prob_gb))

pd.DataFrame(resultados).set_index("modelo")

In [ ]:
# --- Modelo 4: Rede Neural Densa (TensorFlow / Keras) ---
# class_weight para compensar desbalanceamento
n_pos = y_train.sum(); n_neg = len(y_train) - n_pos
class_weight = {0: len(y_train)/(2*n_neg), 1: len(y_train)/(2*n_pos)}

tf.random.set_seed(SEED)
mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURES),)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1,  activation="sigmoid"),
])
mlp.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                         restore_best_weights=True)
hist = mlp.fit(X_train_s, y_train, validation_split=0.15, epochs=120,
               batch_size=32, class_weight=class_weight,
               callbacks=[early], verbose=0)

prob_mlp = mlp.predict(X_test_s, verbose=0).ravel()
p_mlp = (prob_mlp >= 0.5).astype(int)
resultados.append(avaliar("Rede Neural (MLP)", y_test, p_mlp, prob_mlp))

print(f"Treino encerrado em {len(hist.history['loss'])} épocas (EarlyStopping).")
pd.DataFrame(resultados).set_index("modelo")

In [ ]:
# Curvas de aprendizado da MLP — diagnóstico de overfitting
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist.history["loss"], label="treino")
ax[0].plot(hist.history["val_loss"], label="validação")
ax[0].set_title("Perda (binary crossentropy)"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="treino")
ax[1].plot(hist.history["val_accuracy"], label="validação")
ax[1].set_title("Acurácia"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.show()

**Análise:** As curvas de treino e validação caem juntas e estabilizam, sem
divergência — não há sinal de overfitting. A curva de validação fica inclusive
ligeiramente **acima** da de treino, comportamento normal causado pelo *dropout* (ativo
apenas no treino, o que penaliza a métrica de treino). O `EarlyStopping` interrompeu o
treinamento por volta de 40 épocas, restaurando os melhores pesos. O modelo está num regime
saudável: aprende o sinal disponível sem decorar o ruído do treino.

## 9. Otimização de hiperparâmetros

Ajustamos a **Random Forest** com `GridSearchCV`. Como é série temporal, usamos
`TimeSeriesSplit` na validação cruzada — cada *fold* treina no passado e valida no
futuro, **sem vazamento temporal**.

**Hiperparâmetros ajustados e por quê:**
- `n_estimators` — número de árvores (mais árvores = mais estável, até saturar);
- `max_depth` — profundidade máxima (controla complexidade/overfitting);
- `min_samples_leaf` — mínimo de amostras por folha (regulariza).

**Critério de seleção:** F1 (equilíbrio entre precisão e recall), mais informativo
que acurácia sob desbalanceamento.

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

grid = {
    "n_estimators":     [200, 400],
    "max_depth":        [6, 10, None],
    "min_samples_leaf": [1, 5, 10],
}

gs = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1),
    grid, scoring="f1", cv=tscv, n_jobs=-1)
gs.fit(X_train_s, y_train)

print("Melhores hiperparâmetros:", gs.best_params_)
print(f"Melhor F1 (validação cruzada temporal): {gs.best_score_:.3f}")

rf_tuned = gs.best_estimator_
p_rf_t = rf_tuned.predict(X_test_s)
prob_rf_t = rf_tuned.predict_proba(X_test_s)[:, 1]
resultados.append(avaliar("Random Forest (otimizada)", y_test, p_rf_t, prob_rf_t))

pd.DataFrame(resultados).set_index("modelo")

**Análise:** A melhor configuração encontrada foi `max_depth=10`,
`min_samples_leaf=5`, `n_estimators=200`, com F1 em torno de 0,63 na validação cruzada
temporal. O ajuste **melhorou** a Random Forest em relação à versão padrão: o F1 no teste
subiu de ~0,59 para ~0,63, principalmente por equilibrar o recall (a versão padrão era
precisa mas perdia metade das chuvas). Isso confirma que limitar a profundidade e exigir
um mínimo de amostras por folha reduziu o sobreajuste e tornou o modelo mais sensível aos
eventos de chuva.

## 10. Avaliação dos resultados

Consolidamos todos os modelos e analisamos o vencedor em profundidade.

**Métricas escolhidas e justificativa:**
- **Acurácia** — visão geral, mas enganosa sob desbalanceamento.
- **Precisão** — dos dias previstos como "chuva", quantos choveram (evita falsos alarmes).
- **Recall** — dos dias que choveram, quantos o modelo pegou (evita perder eventos).
- **F1** — média harmônica de precisão e recall; **métrica principal** aqui.
- **AUC** — capacidade de ranquear chuva vs. não chuva, independente do limiar.

In [ ]:
tabela = pd.DataFrame(resultados).set_index("modelo").sort_values("f1", ascending=False)
print("Ranking por F1:\n")
tabela.style.format("{:.3f}").background_gradient(cmap="Greens", subset=["f1","auc"])
tabela

In [ ]:
# Seleciona automaticamente o melhor modelo de ML por F1 (exclui baselines)
ml_rows = tabela[~tabela.index.str.contains("Baseline")]
melhor_nome = ml_rows["f1"].idxmax()

mapa_pred = {
    "Regressão Logística": (p_logit, prob_logit),
    "Random Forest": (p_rf, prob_rf),
    "Gradient Boosting": (p_gb, prob_gb),
    "Rede Neural (MLP)": (p_mlp, prob_mlp),
    "Random Forest (otimizada)": (p_rf_t, prob_rf_t),
}
melhor_pred, melhor_prob = mapa_pred[melhor_nome]
print("Melhor modelo por F1:", melhor_nome)

# Matriz de confusão do melhor modelo
cm = confusion_matrix(y_test, melhor_pred)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=["Não chove","Chove"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title(f"Matriz de confusão — {melhor_nome}")

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, melhor_prob)
ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, melhor_prob):.3f}")
ax[1].plot([0,1],[0,1],"--", color="gray")
ax[1].set_xlabel("Falso positivo"); ax[1].set_ylabel("Verdadeiro positivo")
ax[1].set_title("Curva ROC"); ax[1].legend()
plt.tight_layout(); plt.show()

print("\nRelatório de classificação detalhado:\n")
print(classification_report(y_test, melhor_pred, target_names=["Não chove","Chove"], zero_division=0))

### 10.1. Análise de erros e discussão

**Os melhores modelos não lineares ficam num patamar próximo** (F1 ≈ 0,60–0,65,
AUC ≈ 0,75–0,78), com a **Rede Neural (MLP)** e a **Random Forest otimizada** tipicamente
no topo e o Gradient Boosting na mesma vizinhança. A seleção automática (célula acima)
escolhe, a cada execução, o modelo com o maior F1 — que pode alternar entre os primeiros
colocados, dada a proximidade.

> **Sobre a alternância do vencedor.** Os modelos de árvore são determinísticos
> (`random_state` fixo), mas a Rede Neural parte de uma inicialização estocástica. Como os
> melhores modelos têm desempenho quase idêntico, a posição de topo pode alternar entre
> execuções por margens de ~1–2 pontos de F1. Isso não enfraquece a conclusão: significa
> que **famílias de modelos muito diferentes convergiram para o mesmo patamar de
> desempenho**, o que é um forte indício de que esse patamar reflete o limite do sinal
> disponível, e não uma peculiaridade de um modelo.

- **Ganho real sobre o baseline.** O baseline de persistência ("amanhã = hoje") atingiu
  F1 = 0,578. Os melhores modelos o superam de forma consistente (~5–7 pontos de F1) — um
  ganho modesto, mas que comprova que os modelos de ML agregam valor sobre a regra ingênua. Já o baseline de classe majoritária teve F1 = 0,000
  (acurácia 0,597): acerta quase 60% chutando sempre "não chove", sem prever uma única
  chuva. Esse contraste é a prova concreta de que **acurácia isolada é enganosa** neste
  problema e de que o F1 é a métrica correta.

- **Trade-off precisão × recall (os dois perfis).** Os modelos têm "personalidades"
  diferentes. A **Random Forest otimizada** é mais cautelosa: precisão maior e menos
  alarmes falsos (em torno de 53 falsos positivos), mas recall um pouco menor (~0,63),
  perdendo mais dias de chuva. A **MLP** é mais sensível: recall maior (~0,67–0,70),
  capturando mais chuvas, ao custo de mais alarmes falsos. **Para a aplicação contratual do
  projeto, prever chuva a mais (recall alto) costuma ser preferível a perder uma chuva
  real** — então, na prática, mesmo quando a RF otimizada vence por F1, a MLP pode ser a
  escolha de negócio preferível. A decisão final depende do custo relativo de falso
  positivo vs. falso negativo.

- **Leitura da matriz de confusão.** Dos 147 dias de chuva no teste, o melhor modelo acerta
  entre ~90 e ~100 (recall de 63–70%) e perde o restante; dos 218 dias secos, classifica
  corretamente entre ~150 e ~165, gerando da ordem de 50–67 falsos alarmes. É um modelo
  equilibrado, que não colapsa numa única classe.

- **Overfitting / underfitting.** As curvas de aprendizado (seção 8) não mostram
  overfitting. O fato de a Regressão Logística (linear) ficar próxima dos modelos não
  lineares — e de duas famílias distintas empatarem no topo — sugere que o limite está no
  **sinal disponível**, não na capacidade dos modelos: prever chuva a 1 dia com dados
  diários agregados é intrinsecamente difícil.

- **Limitações dos resultados.** (1) dados diários perdem dinâmica intradiária;
  (2) ausência de variáveis sinóticas de larga escala (frentes, sistemas de pressão
  regionais); (3) horizonte de apenas 1 dia; (4) uma única localidade.

- **Melhorias futuras.** Usar dados horários; incluir defasagens de vários dias (janelas
  deslizantes); testar modelos de sequência (LSTM/GRU); incorporar a própria *previsão*
  operacional do Open-Meteo como feature; calibrar o limiar de decisão conforme o custo
  real de falsos positivos vs. negativos no contexto contratual.


## 11. Respondendo nossas hipóteses

Após toda a etapa de visualização, modelagem e avaliação, podemos responder
concretamente às hipóteses levantadas no início do notebook.

### Hipótese 1: A umidade do dia é o preditor isolado mais forte para a chuva do dia seguinte?
**Não — hipótese refutada.** A matriz de correlação (seção 4) mostrou que a umidade tem
correlação de apenas +0,15 com `rain_tomorrow`, **abaixo** da pressão à superfície (−0,31),
da temperatura mínima (+0,27), da precipitação (+0,24) e da temperatura média (+0,23).
Portanto, ao contrário do que eu supunha, a umidade **não** é o preditor isolado mais
forte; a **pressão** (relação inversa: pressão mais baixa → mais chuva) e a **temperatura
mínima** são mais informativas. Refutar uma hipótese com evidência é um resultado tão
válido quanto confirmá-la, e ajustou minha compreensão do problema.

### Hipótese 2: Existe um padrão sazonal claro, com meses sistematicamente mais chuvosos?
**Sim — confirmada.** O gráfico de frequência de chuva por mês (seção 4) evidenciou
variação clara ao longo do ano, indo de ~26% de dias de chuva em julho a ~48% em janeiro.
Essa sazonalidade justificou as features de seno e cosseno do dia do ano, que entraram
entre as variáveis dos modelos.

### Hipótese 3: A persistência ("se choveu hoje, tende a chover amanhã") é um baseline difícil de superar?
**Sim — confirmada.** O baseline de persistência atingiu F1 = 0,578, ficando à frente da
Regressão Logística (~0,60 — empate técnico) e da Random Forest padrão, e só foi superado
com folga pelos dois melhores modelos (Rede Neural e Random Forest otimizada, ambos com
F1 ≈ 0,63–0,65). Isso confirma que a inércia do clima carrega boa parte da informação
disponível a um dia de horizonte, e que superá-la exige um modelo capaz de capturar
relações não lineares.


In [ ]:
# Tempo total de execução e recursos computacionais
elapsed = (time.time() - _T0) / 60
print(f"Tempo total de execução do notebook até aqui: {elapsed:.1f} minutos")
print("Recursos: ambiente padrão do Google Colab (CPU), sem GPU.")

## 12. Conclusão do MVP

**Problema abordado.** Classificação binária de ocorrência de chuva no dia seguinte
em Lajeado/RS, motivada pelo ajuste automático de prazos contratuais sensíveis ao
clima (plataforma Ricarol, do meu TCC).

**Dataset.** 2.708 observações diárias reais (2019–2026) da API pública Open-Meteo,
carregadas por URL, sem upload nem chave.

**Tratamentos principais.** Definição do alvo como chuva > 1 mm no dia seguinte;
engenharia de sazonalidade (seno/cosseno); features de memória de curto prazo;
normalização ajustada só no treino; divisão **temporal** treino/teste; `class_weight`
para o desbalanceamento.

**Modelos avaliados.** Dois baselines (persistência e classe majoritária), Regressão
Logística, Random Forest (padrão e otimizada via GridSearch com `TimeSeriesSplit`) e
uma Rede Neural Densa em TensorFlow.

**Melhor resultado.** Os modelos não lineares mais fortes — **Rede Neural (MLP)**,
**Random Forest otimizada** e **Gradient Boosting** — ficam num patamar próximo
(F1 ≈ 0,60–0,65, AUC ≈ 0,75–0,78), tipicamente com a MLP e a RF otimizada no topo, todos
superando o baseline de persistência (F1 = 0,578). A seleção automática escolhe o de melhor
F1 em cada execução. A escolha pelo F1 se justifica pelo desbalanceamento (61,8% / 38,2%):
queremos equilíbrio entre não dar falsos alarmes (precisão) e não perder eventos de chuva
(recall). No contexto contratual, vale notar que a MLP tende a ter recall mais alto
(captura mais chuvas), enquanto os modelos de árvore tendem a ser mais precisos (menos
alarmes falsos) — a escolha de negócio depende do custo relativo de cada tipo de erro.

> *Observação sobre reprodutibilidade:* os modelos de árvore são determinísticos, mas a
> Rede Neural parte de uma inicialização estocástica; por isso o vencedor por F1 pode
> alternar entre os primeiros colocados a cada execução, por margens de ~1–2 pontos. O fato
> de famílias de modelos distintas convergirem ao mesmo patamar reforça que esse é o limite
> real do sinal disponível.

**O MVP cumpriu o objetivo?** Sim: entregou um fluxo completo, reprodutível e honesto, com
um modelo que supera o baseline de forma transparente e com avaliação crítica das
limitações. O ganho sobre a persistência é modesto (~7 pontos de F1) — o que é o retrato
fiel da dificuldade real de prever chuva a um dia com dados diários agregados.

**Cuidado metodológico central.** O alvo é a chuva **futura** (do dia seguinte), nunca
observada entre as features; combinado ao split temporal e ao scaler ajustado só no treino,
isso garante que as métricas reflitam a capacidade real de generalização, sem vazamento de
dados.

**Próximos passos.** Dados horários, modelos de sequência (LSTM), janelas de defasagem
mais longas, uso da previsão operacional como feature e calibração do limiar conforme o
custo de negócio.


---
## Apêndice — Respostas ao checklist do MVP (resumo)

**Definição do problema.** Classificação binária supervisionada (chove/não chove amanhã);
objetivo: apoiar decisões contratuais sensíveis ao clima; tratável por ML por ser uma
relação não linear e ruidosa aprendida de exemplos. Premissa: histórico recente informa o
curto prazo.

**Dados.** Open-Meteo (ERA5), URL pública, 2.708 registros, 7 variáveis meteorológicas
brutas + features derivadas; alvo `rain_tomorrow`. Limitação: resolução diária e localidade única.

**Preparação.** Sem valores ausentes relevantes; criação de `sin/cos` e features de memória;
normalização ajustada só no treino; **sem features sintéticas**. Vazamento tratado em dois
níveis: alvo no futuro e scaler ajustado só no treino.

**Divisão.** Treino/teste **temporal** (últimos 12 meses como teste); validação cruzada
temporal (`TimeSeriesSplit`) no tuning. Ordem cronológica respeitada.

**Modelagem.** Baselines (persistência, classe majoritária) + Logística + Random Forest +
Gradient Boosting + MLP. Comparação justa nas mesmas partições.

**Otimização.** GridSearch na Random Forest (`n_estimators`, `max_depth`, `min_samples_leaf`),
critério F1, sem tocar no teste.

**Avaliação.** Acurácia, precisão, recall, F1 (principal), AUC, matriz de confusão, ROC,
análise de erros e de overfitting.

**Conclusão.** Melhor modelo escolhido por F1; objetivo cumprido; limitações e próximos
passos descritos acima.
